# 环节 01 · 带宽天花板与量化档位（配套 Notebook）

> 配套：[benchmark.md](./benchmark.md) §一 / §五 / §六 · 导航：[环节00](./环节00-总揽与环节导航.md)
> 原理对照：[环节10 演示](../../foundation/transformer/环节10-推理解码与KV缓存演示.ipynb)（两阶段性格）、[环节11 演示](../../foundation/transformer/环节11-服务化与推理引擎演示.ipynb)（容量账）
> 定位：把「decode 速度由内存带宽决定」算成可改参数的数。**纯标准库，不调本机模型。**

**怎么跑**：逐格 `Shift+Enter`。换机型只改 §1 的 `BW_GBS`。

| 本 Notebook | 手册 | 验证什么 |
|---|---|---|
| §1 理想公式 | README §三、benchmark §一 | `tok/s ≈ BW / W` |
| §2 小模型截距 | benchmark §六 | `t ≈ 1.6 + W/261`，0.6B 会高估 2 倍 |
| §3 有效带宽 | benchmark §六表 | 模型越大越接近峰值 |
| §4 量化只治 decode | benchmark §五 | prefill 几乎不动，decode 随体积反比 |
| §5 换机型 | Apple 规格 | 120 / 273 / 410 GB/s 整体平移 |


## 1. 理想公式：decode 每步要把权重整体搬一遍

Decode 每生成 1 个 token，就要把**整份权重**从统一内存读过一遍（再加一点点 KV）。所以粗估：

```
decode tok/s ≈ 内存带宽 (GB/s)  ÷  每 token 读取的权重大小 (GB)
```

本机（M4 Pro）官方带宽 **273 GB/s**。下面用 [benchmark.md](./benchmark.md) 里 Qwen3-0.6B 的实测体积。


In [ ]:
BW_GBS = 273.0  # M4 Pro；普通 M4=120，M4 Max=410。换机型只改这里。

WEIGHTS = {
    "MLX 4-bit": 0.336,
    "GGUF Q4_K_M": 0.517,
    "MLX 8-bit": 0.633,
    "MLX bf16": 1.192,
    "GGUF F16": 1.503,
    "Ollama qwen3.5:9b Q4_K_M": 6.6,
}

MEASURED_DECODE = {
    "MLX 4-bit": 349.3,
    "GGUF Q4_K_M": 282.8,
    "MLX 8-bit": 257.7,
    "MLX bf16": 162.8,
    "GGUF F16": 149.7,
    "Ollama qwen3.5:9b Q4_K_M": 38.6,
}

print(f"带宽 {BW_GBS:g} GB/s\n")
print(f"{'场景':<28} {'W GB':>7} {'理想 tok/s':>12} {'实测':>8} {'高估':>8}")
for name, w in WEIGHTS.items():
    ideal = BW_GBS / w
    meas = MEASURED_DECODE[name]
    print(f"{name:<28} {w:>7.3f} {ideal:>12.1f} {meas:>8.1f} {ideal/meas:>7.2f}×")


## 2. 小模型的真相：固定开销 1.6 ms/token

理想公式在 0.6B 4-bit 上给出 812 tok/s，实测只有 349——高估 **2.3 倍**。

每次 forward 有一笔**与权重大小无关**的开销（kernel 启动、采样、运行时循环）。本机三档 MLX 拟合：

```
t_ms ≈ 1.6 + W_GB / 261
```

斜率 261 ≈ 273 的 96% → **带宽天花板是对的**；截距才是小模型被高估的原因。


In [ ]:
INTERCEPT_MS = 1.6
SLOPE_GBS = 261.0  # 有效带宽，≈ 官方 273 的 96%


def predict_toks(w_gb: float) -> tuple[float, float]:
    t_ms = INTERCEPT_MS + 1000.0 * w_gb / SLOPE_GBS
    return t_ms, 1000.0 / t_ms


print(f"{'场景':<28} {'预测 ms':>8} {'预测 tok/s':>12} {'实测':>8} {'误差':>8}")
for name, w in WEIGHTS.items():
    t_ms, toks = predict_toks(w)
    meas = MEASURED_DECODE[name]
    err = (toks - meas) / meas * 100
    print(f"{name:<28} {t_ms:>8.2f} {toks:>12.1f} {meas:>8.1f} {err:>+7.1f}%")

print()
w = 0.336
naive = BW_GBS / w
_, fitted = predict_toks(w)
print(f"0.6B 4-bit：理想 {naive:.0f} vs 拟合 {fitted:.1f} vs 实测 {MEASURED_DECODE['MLX 4-bit']}")
print("→ 1B 以下不要用「带宽÷权重」做容量承诺；7B+ 才开始靠谱。")


## 3. 有效带宽 = 权重 × 实测 decode（看离峰值多远）


In [ ]:
print(f"峰值 {BW_GBS:g} GB/s\n")
print(f"{'场景':<28} {'有效 GB/s':>10} {'占峰值':>8}")
for name, w in WEIGHTS.items():
    eff = w * MEASURED_DECODE[name]
    print(f"{name:<28} {eff:>10.0f} {eff/BW_GBS:>7.0%}")

print()
print("规律：模型越大，有效带宽越接近峰值。")
print("推论：同一台机器上，模型越大，『换引擎』能拉开的差距越小——都被带宽压平。")


## 4. 量化只加速 decode，prefill 几乎不动

[benchmark.md §五](./benchmark.md) 同模型 MLX 三档。prefill 是算力受限，decode 是带宽受限。


In [ ]:
QUANT = [
    # bits_per_w, weight_gb, prefill, decode
    (16.0, 1.192, 5521.7, 162.8),
    (8.50, 0.633, 5629.1, 257.7),
    (4.00, 0.336, 5568.4, 349.3),
]
base_pre, base_dec = QUANT[0][2], QUANT[0][3]
print(f"{'档位':>6} {'W GB':>7} {'prefill':>10} {'相对':>8} {'decode':>8} {'相对':>8}")
for bits, w, pre, dec in QUANT:
    print(f"{bits:>5.1f}b {w:>7.3f} {pre:>10.1f} {pre/base_pre:>7.2f}× {dec:>8.1f} {dec/base_dec:>7.2f}×")

pres = [p for _, _, p, _ in QUANT]
print(f"\nprefill 波动 {(max(pres)-min(pres))/min(pres)*100:.1f}%  → 量化几乎不治 prefill")
print(f"decode 4-bit / bf16 = {QUANT[2][3]/QUANT[0][3]:.2f}×  → 这才是『量化提速』的全部机制")


## 5. 换机型：绝对值平移，结构不变

把 §2 的截距留着（kernel 开销跟带宽关系不大），只改带宽斜率。


In [ ]:
MACHINES = [("普通 M4", 120.0), ("M4 Pro（本机）", 273.0), ("M4 Max", 410.0)]
W_DEMO = 0.336  # 0.6B 4-bit

print(f"同一份 {W_DEMO} GB 权重：\n")
for name, bw in MACHINES:
    # 斜率按本机 261/273 的效率折一下
    slope = bw * (SLOPE_GBS / BW_GBS)
    t_ms = INTERCEPT_MS + 1000.0 * W_DEMO / slope
    print(f"  {name:<14} {bw:>4.0f} GB/s  →  {1000/t_ms:6.0f} tok/s  ({t_ms:.2f} ms/tok)")

print("\n换机型不要直接套本手册的 349 tok/s；套公式，或至少按带宽比例缩放大模型数字。")
